# Domain feature packs end to end

AFX ships four presets, validated configs that `AFX.load_preset(name)` returns and any AFX entry point accepts. Set `DOMAIN` in the config cell and run all: the notebook downloads a small open dataset, extracts the preset's features, aggregates them, trains a scikit-learn classifier and plots accuracy, a confusion matrix and two domain-specific views.

| `DOMAIN` | Preset | Dataset | Task |
|---|---|---|---|
| `speech` | Speech & paralinguistics | RAVDESS (speech audio) | emotion classification |
| `bioacoustics` | Bioacoustics | ESC-50 (animal subset) | animal sound classification |
| `music` | Music & MIR | GTZAN | genre classification |
| `machinery` | Machinery & condition monitoring | DCASE 2020 Task 2 dev (pump) | normal vs anomaly |

Datasets download into `$DATASETS` (default `./datasets`). With the default per-class subsets everything stays under 4 GB and each domain runs in a few minutes on a CPU.

Requires `pip install -e ".[notebooks]"`.

## 1. Configuration

Edit this cell, then **Run All**. Everything downstream keys off `DOMAIN`.

In [ ]:
import os, sys, time
import numpy as np

# Make the demo helper importable (it lives next to this notebook).
sys.path.insert(0, os.getcwd())
import domain_packs as dp
from AFX.presets import load_preset, preset_info

# ---- choose your domain -------------------------------------------------- #
DOMAIN = "speech"   # "speech" | "bioacoustics" | "music" | "machinery"

# ---- knobs (sensible defaults; raise for a fuller run) ------------------- #
N_PER_CLASS  = 60      # clips sampled per class (None = use all)
CLASSES      = None    # None = the preset's default class subset; or a list
MAX_SECONDS  = 8.0     # truncate each clip for speed (None = full length)
TEST_SIZE    = 0.30
RANDOM_STATE = 0

assert DOMAIN in dp.DOMAINS, f"DOMAIN must be one of {list(dp.DOMAINS)}"
spec    = dp.DOMAINS[DOMAIN]
classes = list(CLASSES) if CLASSES else (list(spec.default_classes)
                                         if spec.default_classes else None)
print(f"Domain      : {spec.title}")
print(f"Preset pack : {spec.preset}")
print(f"Task        : {spec.task}")
print(f"Dataset     : {spec.dataset_name}  (~{spec.approx_download_mb} MB download)")
print(f"Classes     : {classes}")

## 2. Inspect the preset pack

A preset is just a dict: a `sample_rate`, a curated `features` block with domain-tuned frame/hop sizes, and an `aggregation` strategy.

In [ ]:
config = load_preset(spec.preset)
info   = preset_info(spec.preset)

print(info["title"] + ":", info["description"], "\n")
print(f"sample_rate : {config['sample_rate']} Hz")
print(f"aggregation : {config['aggregation']}")
print(f"features    ({len(config['features'])}):")
for name, params in config["features"].items():
    print(f"    - {name:20s} {params}")

## 3. Download & prepare the dataset

Idempotent: the archive is fetched once into `$DATASETS` and skipped on subsequent runs. The first run of a large domain (GTZAN, DCASE) can take a while.

In [ ]:
root = dp.datasets_root()
print("datasets root:", root)

t0 = time.time()
data_dir = spec.prepare(root)
examples = spec.collect(data_dir, classes, N_PER_CLASS, RANDOM_STATE)
print(f"\nprepared in {time.time()-t0:.1f}s, {len(examples)} clips")

from collections import Counter
for label, n in sorted(Counter(l for _, l in examples).items()):
    print(f"    {label:12s} {n}")

## 4. Extract preset features

`extract_all_features` applies the preset's per-feature params **and** its aggregation, so every clip becomes one fixed-length descriptor vector.

In [ ]:
t0 = time.time()
X, y, feature_names = dp.build_feature_matrix(
    examples, config, max_seconds=MAX_SECONDS, verbose=True)
print(f"\nfeature matrix: X={X.shape}  ({len(feature_names)} features)  "
      f"in {time.time()-t0:.1f}s")
print("first columns:", feature_names[:8], "...")

## 5. Train a small classifier

Standardized features and an untuned random forest, enough to show the pack carries signal.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

clf = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(f"accuracy: {accuracy_score(y_te, y_pred):.3f}  "
      f"(chance ≈ {1/len(set(y)):.3f})\n")
print(classification_report(y_te, y_pred))

## 6. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

labels = sorted(set(y))
fig, ax = plt.subplots(figsize=(1.4*len(labels)+2, 1.4*len(labels)+1))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred, labels=labels, cmap="Blues", colorbar=False,
    xticks_rotation=45, ax=ax)
ax.set_title(f"{spec.title}: confusion matrix")
plt.tight_layout(); plt.show()

## 7. Which features carry the signal?

Random-forest importances show which parts of the pack the model used for this task.

In [ ]:
rf = clf.named_steps["randomforestclassifier"]
imp = rf.feature_importances_
order = np.argsort(imp)[::-1][:15]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(order)), imp[order][::-1], color="#4c72b0")
ax.set_yticks(range(len(order)))
ax.set_yticklabels([feature_names[i] for i in order][::-1], fontsize=9)
ax.set_xlabel("importance"); ax.set_title(f"{spec.title}: top 15 features")
plt.tight_layout(); plt.show()

## 8. Feature-space view (PCA)

Project the standardized descriptors to 2-D to *see* how separable the classes are with this pack.

In [ ]:
from sklearn.decomposition import PCA

Xs = StandardScaler().fit_transform(X)
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xs)

fig, ax = plt.subplots(figsize=(7, 6))
for label in sorted(set(y)):
    m = y == label
    ax.scatter(pcs[m, 0], pcs[m, 1], s=18, alpha=0.7, label=label)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title(f"{spec.title}: PCA of preset features")
ax.legend(fontsize=8, markerscale=1.5); plt.tight_layout(); plt.show()

## Switch domains, or build your own pack

**Try another domain:** change `DOMAIN` in cell 1 to `bioacoustics`, `music`,
or `machinery` and Run All.

**Roll your own pack:** a preset is just a JSON file in `AFX/presets/`. Copy one,
pick the AFX features that matter for your problem, tune the frame/hop sizes and
`aggregation`, and it's immediately loadable:

```python
from AFX.presets import list_presets, load_preset
list_presets()                 # ['bioacoustics', 'machinery', 'music', 'speech']
config = load_preset("music")  # -> a validated AFX config dict
```

The same `config` works with `extract_all_features`, the folder-level
`extract_features_from_folder`, and the scikit-learn `AFXFeatureExtractor`.